# 워크숍 3 - 내비게이션: `go_to` 다시 구현하기

**예상 시간:** 약 90분  
**선수 지식:** 워크숍 1 & 2

내장 `go_to` 스킬은 블랙박스입니다. 이번 워크숍에서는 이를 두 가지 방식으로 직접 만들어 봅니다.

- **파트 A - 전역 상태 기반 내비게이션:** `scene_state`와 `robot_status`의 자세 데이터를 사용해 heading error를 계산하고 비례 제어기로 주행합니다.
- **파트 B - 비전 전용 내비게이션:** 특권 위치 데이터 없이 로봇 카메라(색상 감지 + blob 면적)만 사용해 이동합니다.

두 접근법을 모두 이해하는 것은 GPS나 정확한 위치 추정이 없을 수 있는 실제 로봇 환경에서 중요합니다.

> **워크숍 기본 가정:** 아래 예제는 목표가 보이고 직접 경로가 장애물로 막히지 않은 통제된 설정에서 실행하도록 설계되었습니다. 막힌 경로, 목표 상실, 능동 스캔, 장애물 회피 우회는 여기서 개념적으로 소개하고 프로젝트/심화 과제로 이어집니다.


> **시작하기 전에:** 이전 워크숍에서 열어 둔 시뮬레이터 창을 모두 닫으세요. 각 노트북은 새 로봇을 만듭니다. 뷰어 탭 두 개가 동시에 열려 있으면 문제가 생길 수 있습니다.


## 섹션 1 - 빠른 설정


In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python matplotlib

In [ ]:
# 워크숍 1과 동일한 설정 — 자세한 설명은 해당 노트북을 참고하세요
import asyncio
import math
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ────────────────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv가 설치되지 않았으므로 .env 모드가 아닙니다

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # 키가 이미 환경 변수에 설정되어 있습니다(CI, Docker 등)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")


In [ ]:
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop3-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> 뷰어 URL을 **Google Chrome**에서 열고 장면이 로드될 때까지 기다린 다음, 아래 셀을 실행하세요.


In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # 아직 뷰어가 접속하지 않았습니다
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")


---
# 파트 A - 전역 상태 기반 내비게이션

파트 A에서는 `scene_state`와 `robot_status`의 world coordinate를 사용해 이동합니다. 가장 안정적인 접근법이지만 시뮬레이션의 특권 데이터가 필요합니다.


## 파트 A 설정: `pad_C`까지 열린 경로 만들기

섹션 2를 시작하기 전에 Chrome 뷰어를 사용해 로봇과 `pad_C` 사이에 대체로 직선이고 장애물이 없는 경로가 생기도록 로봇을 이동시키세요. 로봇이 아직 `pad_C`를 바라볼 필요는 없습니다. 섹션 3에서 heading error를 계산하고 로봇을 돌립니다.

이 설정은 파트 A를 좌표 기하와 비례 제어에 집중하게 해줍니다. 다른 물체가 직선 경로를 막고 있으면 단순한 `drive_to_distance` baseline은 장애물을 계획적으로 피하지 못하므로 멈추거나 목표에서 멀리 떨어진 곳에 정지할 수 있습니다.


## 섹션 2 - 로봇 자세와 world coordinate

창고는 2D 좌표계를 사용합니다.
- **yaw = 0도** -> 로봇이 +x 방향을 바라봄
- **yaw = 90도** -> 로봇이 +y 방향을 바라봄
- **bearing** = `atan2(dy, dx)` - world frame에서 목표까지의 각도
- **heading_error** = 목표를 바라보기 위해 로봇이 몇 도 돌아야 하는지


In [ ]:
# 로봇 자세와 목표 위치를 읽습니다
status = await session.state.get("robot_status")
scene  = await session.state.get("scene_state")

rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
yaw    = status.robot.pose.yaw_deg

pad_c  = scene.entities["pad_C"]
tx, ty = pad_c.pose.position[0], pad_c.pose.position[1]

print(f"Robot   : x={rx:.2f}, y={ry:.2f}, yaw={yaw:.1f}°")
print(f"pad_C   : x={tx:.2f}, y={ty:.2f}")

# 기하 계산
dist          = math.hypot(tx - rx, ty - ry)
bearing       = math.degrees(math.atan2(ty - ry, tx - rx))
heading_error = (bearing - yaw + 180) % 360 - 180

print(f"\nDistance      : {dist:.2f} m")
print(f"Bearing       : {bearing:.1f}° (world frame)")
print(f"Heading error : {heading_error:.1f}° (how much to turn)")


## 섹션 3 - 목표를 바라보도록 회전하기


In [ ]:
async def turn_to_face(session, target_pos):
    """target_pos = [x, y, ...]를 바라보도록 로봇을 회전시킵니다."""
    state = await session.state.get("robot_status")
    rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
    yaw = state.robot.pose.yaw_deg
    tx, ty = target_pos[0], target_pos[1]

    bearing = math.degrees(math.atan2(ty - ry, tx - rx))
    heading_error = (bearing - yaw + 180) % 360 - 180
    print(f"  Heading error: {heading_error:+.1f}°")

    if abs(heading_error) < 5:
        print("  Already facing target.")
        return

    # wz > 0이면 왼쪽, wz < 0이면 오른쪽으로 회전합니다
    wz = 0.4 if heading_error > 0 else -0.4
    # duration = angle / angular_speed  (라디안 기준)
    duration = min(abs(heading_error) / (abs(wz) * 57.296), 8.0)

    await session.invoke(
        "set_velocity",
        {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": duration},
        timeout_s=30,
    )
    print(f"  Turned {heading_error:+.0f}° in {duration:.2f}s")


# 시연: pad_C를 바라보도록 회전합니다
scene = await session.state.get("scene_state")
pad_c_pos = scene.entities["pad_C"].pose.position
print("Turning to face pad_C...")
await turn_to_face(session, pad_c_pos)


## 섹션 4 - 목표를 향해 주행하기


In [ ]:
async def drive_to_distance(session, target_pos, tolerance=0.6):
    """target_pos로 전진하여 tolerance 미터 안에 들어올 때까지 이동합니다."""
    for step in range(20):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"  Step {step+1}: dist={dist:.2f}m")
        if dist <= tolerance:
            print("  Arrived!")
            return True
        vx = min(0.8, dist * 0.4)  # 거리에 비례해 속도를 정합니다
        await session.invoke(
            "set_velocity",
            {"vx": vx, "vy": 0.0, "wz": 0.0, "duration_s": 0.8},
            timeout_s=30,
        )
    print("  Max iterations reached.")
    return False


# 시연: pad_C를 바라본 뒤 그쪽으로 주행합니다
print("Driving toward pad_C...")
await drive_to_distance(session, pad_c_pos, tolerance=0.8)


> **Baseline 한계:** `drive_to_distance`는 목표 좌표를 향해 주행하지만 다른 물체를 피해 경로를 계획하지 않습니다. 직선 경로가 막히면 로봇이 걸리거나 목표에서 멀리 멈출 수 있습니다. 이 워크숍의 필수 예제는 경로가 열린 설정에서 실행하세요.


## 섹션 5 - 완성된 `my_go_to_global`

회전과 주행을 반복 보정 루프로 결합합니다.


In [ ]:
async def my_go_to_global(session, entity_id, tolerance=0.6, max_iters=10):
    """scene_state와 robot_status를 사용해 entity_id까지 이동합니다."""
    scene = await session.state.get("scene_state")
    entity = scene.entities.get(entity_id)
    if entity is None:
        print(f"Entity '{entity_id}' not found in scene.")
        return False
    target_pos = entity.pose.position

    for i in range(max_iters):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"Iter {i+1}: dist={dist:.2f}m to {entity_id}")
        if dist <= tolerance:
            print(f"Reached {entity_id}!")
            return True
        await turn_to_face(session, target_pos)
        await drive_to_distance(session, target_pos, tolerance=tolerance)

    print("Max iterations reached.")
    return False


# 데모: pad_C까지 이동합니다
print("my_go_to_global → pad_C")
success = await my_go_to_global(session, "pad_C")
print(f"Result: {success}")


---
# 파트 B - 비전 전용 내비게이션

실제 세계의 로봇은 정확한 자세 데이터에 접근하지 못할 수 있습니다. 파트 B에서는 `scene_state`도 `robot_status`도 사용하지 않고 로봇 카메라만 사용해 이동합니다.


## 섹션 6 - 비전 전용 방식이 중요한 이유

- **시뮬레이션에서 실제로 이전:** 시뮬레이션과 실제 하드웨어 사이에는 차이가 있으므로 정확한 좌표 같은 특권 데이터는 실제 로봇에 거의 없습니다.
- **보편적인 센서:** 카메라는 가장 흔한 온보드 센서이며, 전역 정보가 항상 제공되는 것은 아닙니다.
- **도전:** 비전은 노이즈가 있고, 조명이 변하며, 물체가 가려질 수 있으므로 더 견고한 알고리즘이 필요합니다.

핵심 통찰은 "큐브가 world coordinate에서 어디 있나?"가 아니라 "큐브가 내 카메라 화면 어디에 있고, 그 간격을 어떻게 줄일까?"라고 묻는 것입니다.


## 섹션 7 - 독립 실행 가능한 `perceive()` 함수

워크숍 2의 같은 함수를 여기로 복사해 이 노트북만으로 실행 가능하게 만들었습니다.


In [ ]:
# 워크숍 2에서 복사했습니다 — 다른 노트북을 import하지 않아도 실행됩니다
async def perceive(session):
    """보이는 큐브 색상별로 {color: {angle_deg, blob_area}}를 반환합니다."""
    jpeg = await session.get_vision("pov")
    img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    COLOR_RANGES = {
        "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
                   (np.array([160,50, 50]), np.array([180, 255, 255]))],
        "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
        "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
        "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
    }

    result = {}
    for color, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        if area < 200:
            continue
        M = cv2.moments(largest)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        angle_deg = (cx - w / 2) / (w / 2) * 30.0  # HFOV/2 = 30°
        result[color] = {"angle_deg": round(angle_deg, 1), "blob_area": int(area)}
    return result


print(await perceive(session))


## 섹션 8 - 목표 색상을 화면 중앙에 맞추기

목표 색상의 blob이 카메라 중심에서 ±10도 안에 들어올 때까지 로봇을 회전시킵니다.


In [ ]:
async def center_on_color(session, target_color, angle_tolerance=10.0, max_steps=12):
    """target_color 블롭이 카메라 중심 근처에 올 때까지 회전합니다."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible — scanning...")
            await session.invoke(
                "set_velocity",
                {"vx": 0.25, "vy": 0.0, "wz": 0.3, "duration_s": 1.0},
                timeout_s=15,
            )
            continue

        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: {target_color} at {angle:+.1f}°")

        if abs(angle) <= angle_tolerance:
            print("  Target centered!")
            return True

        # 중심을 향해 회전합니다: 양수 각도 = 물체가 오른쪽 → 오른쪽 회전(wz 음수)
        wz = -0.3 if angle > 0 else 0.3
        await session.invoke(
            "set_velocity",
            {"vx": 0.25, "vy": 0.0, "wz": wz, "duration_s": 0.8},
            timeout_s=15,
        )

    print("  Max steps reached.")
    return False


print("Centering on red cube...")
success = await center_on_color(session, "red")
print(f"Result: {success}")


## 섹션 9 설정: 보이는 목표와 열린 경로

`drive_toward_color`를 실행하기 전에 Chrome 뷰어를 사용해 로봇 카메라에 큐브가 하나 이상 보이고, 그 큐브까지의 경로가 대체로 열려 있는 위치로 로봇을 옮기세요. 큐브가 완벽히 중앙에 있을 필요는 없습니다. 섹션 8에서 먼저 중앙에 맞출 수 있습니다.

장애물이 경로를 막거나 목표가 카메라 시야를 벗어나면 baseline은 실패할 수 있습니다. 필수 워크숍 코드가 반드시 해결해야 하는 문제라기보다, 심화/프로젝트 토론을 위한 유용한 관찰로 받아들이세요.


## 섹션 9 - Blob 크기로 주행하기

로봇이 물체에 가까워질수록 이미지 안의 blob이 커집니다. 우리는 blob 면적을 거리의 대략적인 대리값으로 사용합니다. 면적이 임계값을 넘을 때까지 앞으로 주행합니다.

**경험칙:** `ARRIVAL_AREA = 8000 px²`는 로봇이 큐브에 팔이 닿을 정도로 가까워진 상태에 대략 대응합니다.


In [ ]:
ARRIVAL_AREA = 8000  # pixels² — 필요하면 조정하세요

async def drive_toward_color(session, target_color, arrival_area=ARRIVAL_AREA, max_steps=15):
    """블롭 면적을 거리의 대리값으로 사용해 target_color를 향해 전진합니다."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible")
            return False

        area  = obs[target_color]["blob_area"]
        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: area={area}px², angle={angle:+.1f}°")

        if area >= arrival_area:
            print("  Arrived (blob large enough)!")
            return True

        # 경로에서 벗어나지 않도록 두 단계마다 다시 중앙을 맞춥니다
        if step % 2 == 1 and abs(angle) > 10:
            wz = -0.3 if angle > 0 else 0.3
            await session.invoke(
                "set_velocity",
                {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": 0.5},
                timeout_s=15,
            )
        else:
            await session.invoke(
                "set_velocity",
                {"vx": 0.5, "vy": 0.0, "wz": 0.0, "duration_s": 1.0},
                timeout_s=15,
            )

    print("  Max steps reached.")
    return False


print("Driving toward red cube...")
await drive_toward_color(session, "red")


## 섹션 10 - 완성된 `my_go_to_visual`


In [ ]:
async def my_go_to_visual(session, target_color, arrival_area=ARRIVAL_AREA):
    """카메라만 사용해 target_color 큐브까지 이동합니다. scene_state는 사용하지 않습니다."""
    print(f"Navigating to {target_color} cube (vision only)...")
    centered = await center_on_color(session, target_color)
    if not centered:
        print("Could not center on target.")
        return False
    return await drive_toward_color(session, target_color, arrival_area=arrival_area)


# 보이는 큐브 색상으로 데모를 실행합니다
obs = await perceive(session)
if obs:
    demo_color = next(iter(obs))  # 처음 감지된 색상을 선택합니다
    print(f"Demo: navigating to {demo_color}")
    await my_go_to_visual(session, demo_color)
else:
    print("No cubes visible — move the robot closer to the conveyor belt first.")


---
## 추가 질문: 장애물을 고려하는 비전 내비게이션

위 baseline 함수들은 일부러 내비게이션을 단순하게 유지합니다. 직접 경로가 막히거나 목표가 카메라 시야를 벗어났을 때 어떻게 확장할 수 있을지 생각해 보세요.

- `set_head`는 로봇 몸 전체를 바로 돌리지 않고 좌우를 스캔하는 데 어떻게 도움이 될까요?
- 언제 로봇이 목표를 다시 중앙에 맞추기 전에 뒤로 물러나야 할까요?
- 로봇은 마지막으로 본 목표 방향을 어떻게 기억할 수 있을까요?
- 로봇이 단순히 천천히 움직이는 것이 아니라 장애물 뒤에 막혔다는 증거는 무엇일까요?

이 질문들은 주요 워크숍 연습의 필수 사항은 아닙니다. 고급 프로젝트 레벨과 직접 연결됩니다.


---
## 연습 문제


### 연습 1 - 위치와 거리 출력하기(파트 A)

`scene_state`에서 로봇의 현재 `(x, y)` 위치와 `pad_C`의 위치를 읽으세요. 두 점 사이의 직선 거리를 계산해 출력하세요.


In [ ]:
## 연습 문제 1: 로봇 위치, pad_C 위치, 거리 출력
# 여기에 코드를 작성하세요

# 예상 출력:
# Robot: (1.23, 0.45)
# pad_C: (5.60, 2.10)
# Distance: 4.62 m


### 연습 2 - `pad_C`를 바라보도록 회전하기(파트 A)

`scene_state`에서 `pad_C`의 위치를 가져오고 `turn_to_face(session, pad_c_pos)`를 호출하세요. 회전 전후의 heading error를 출력하세요.


In [ ]:
## 연습 문제 2: pad_C를 바라보도록 회전
# 여기에 코드를 작성하세요

# 예상 출력:
# Before: heading_error = +42.3°
# [회전 실행]
# After: heading_error ≈ 0–5°


### 연습 3 - `pad_C`까지 주행하기(파트 A)

`pad_C`를 바라보도록 회전한 뒤 `drive_to_distance(session, pad_c_pos, tolerance=0.8)`를 호출하세요. 각 반복에서 거리를 출력하세요.


In [ ]:
## 연습 문제 3: pad_C까지 주행
# 여기에 코드를 작성하세요

# 예상 출력:
# Step 1: dist=4.62m
# Step 2: dist=3.45m
# ...
# Arrived!


### 연습 4 - `pad_D`에서 `my_go_to_global` 테스트하기(파트 A)

`my_go_to_global(session, "pad_D")`를 호출하세요. 반환 후 `scene_state`로 `pad_D`까지의 최종 거리를 출력해 도착을 검증하세요.


In [ ]:
## 연습 문제 4: pad_D까지 이동하고 검증
# 여기에 코드를 작성하세요

# 예상 결과: 최종 거리 < 0.6m
# 예: "Arrived at pad_D. Final distance: 0.42m"


### 연습 5(심화) - 경로 비교: `my_go_to_global` vs 내장 `go_to`(파트 A)

`pad_B`까지 `my_go_to_global`로 이동하는 동안과 내장 `go_to`로 이동하는 동안 로봇 위치를 1초마다 기록하세요. `matplotlib`으로 두 경로를 2D 산점도에 그려 비교하세요.


In [ ]:
## 연습 문제 5(심화): pad_B까지의 경로 비교
# 여기에 코드를 작성하세요

# 힌트:
# 1. robot_status를 1초마다 읽어 위치를 리스트에 추가하는 백그라운드 작업을 시작합니다
# 2. my_go_to_global(session, "pad_B")를 실행합니다
# 3. 내장 go_to로 같은 실험을 반복합니다
# 4. 각 경로에 대해 plt.plot(xs, ys)를 그리고 시작/끝 지점과 라벨을 표시합니다


### 연습 6 - 카메라 각도 오프셋 측정하기(파트 B)

`perceive(session)`를 호출하고 빨간 큐브의 `angle_deg`를 출력하세요. 그런 다음 `scene_state`와 `robot_status`로 ground truth 각도를 계산하고 두 값을 비교하세요.


In [ ]:
## 연습 문제 6: 카메라 각도와 scene_state 실제 기준값 비교
# 여기에 코드를 작성하세요

# 예상 출력:
# Camera angle: +14.2°
# Scene truth:  +12.8°
# Error:         1.4°


### 연습 7 - 빨간 큐브를 화면 중앙에 맞추기(파트 B)

`center_on_color(session, "red")`를 호출하세요. 각 단계의 angle offset을 출력하고 0도에 가까워지는 모습을 관찰하세요.


In [ ]:
## 연습 문제 7: 빨간 큐브를 중앙에 맞추기
# 여기에 코드를 작성하세요

# 예상 출력:
# Step 1: red at +18.5°
# Step 2: red at +9.2°
# Step 3: red at +3.1° — Target centered!


### 연습 8 - 빨간 큐브를 향해 주행하기(파트 B)

중앙에 맞춘 뒤 `drive_toward_color(session, "red")`를 호출하세요. 각 단계의 blob 면적을 출력하고 로봇이 접근할수록 면적이 커지는 모습을 관찰하세요.


In [ ]:
## 연습 문제 8: 블롭 크기로 빨간 큐브를 향해 주행
# 여기에 코드를 작성하세요

# 예상 출력:
# Step 1: area=1240px², angle=+2.1°
# Step 2: area=2870px², angle=+1.8°
# ...
# Arrived (blob large enough)!


### 연습 9(심화) - 비전 전용 내비게이션과 실패 분석(파트 B)

파란 큐브가 보이는 통제된 설정에서 `my_go_to_visual(session, "blue")`를 호출하세요. 완료 후 `scene_state`로 파란 큐브까지의 실제 거리를 확인하세요.

그런 다음 목표를 잃거나 경로가 막히는 더 어려운 경우를 하나 시도해 보세요. 무엇이 먼저 실패하나요? 중앙 맞추기, 접근, 목표 가시성, 장애물 처리 중 어디인가요?


In [ ]:
## 연습 문제 9(심화): 비전 전용 내비게이션 + 실패 분석
# 여기에 코드를 작성하세요

# 통제된 설정에서의 예상:
# 내비게이션이 완료되거나 목표 근처까지 접근합니다.
# 파란 큐브까지의 최종 거리: 대략 0.5~1.5m.
#
# 장애물/목표 상실 설정에서의 예상:
# baseline은 실패할 수 있습니다. 어디에서 실패하는지 기록하고,
# set_head 스캔, 뒤로 물러나기, 다시 중앙 맞추기 같은 어떤 복구 행동이 필요한지 적어 보세요.


---
## 정리


In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")